In [1]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

from qiskit_aer.primitives import EstimatorV2 as Estimator

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [2]:
import pandas as pd

n_samples = 2000  # subset — full 155K rows is impractical for QNN

# Shuffle before sampling so we get a mix of fire/no-fire rows
df_full = pd.read_csv("features.csv").sample(n=n_samples, random_state=42).reset_index(drop=True)
df_x    = df_full[["month_sin","month_cos","year","avg_tmax_c","temp_range",
                    "tot_prcp_mm","dryness_3m_avg","latitude","longitude"]]

x = df_x.values.astype(float)
y = df_full["target"].values.astype(float)

# split and scale
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=0)

x_scaler = StandardScaler()
x_train_s = x_scaler.fit_transform(x_train)
x_test_s  = x_scaler.transform(x_test)

angle_scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
x_train_a = angle_scaler.fit_transform(x_train_s)
x_test_a  = angle_scaler.transform(x_test_s)

y_scaler = MinMaxScaler(feature_range=(-1, 1))
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_s  = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

print(f"Train: {x_train_a.shape}, Test: {x_test_a.shape}")
print(f"y_train range: [{y_train_s.min():.3f}, {y_train_s.max():.3f}]")

Train: (1800, 9), Test: (200, 9)
y_train range: [-1.000, 1.000]


In [ ]:
n_features = x_train_a.shape[1]
n_qubits = n_features

x_params = ParameterVector("x", n_features)
reps = 3
n_theta = (n_qubits)
theta_params = ParameterVector("θ", length=n_qubits * 2 * reps)

qc = QuantumCircuit(n_qubits)
for i in range(n_qubits):
    qc.ry(x_params[i], i)

t = 0
for r in range(reps):
    for q in range(n_qubits):
        qc.ry(theta_params[t], q); t += 1
        qc.rz(theta_params[t], q); t += 1

    for q in range(n_qubits - 1):
        qc.cx(q, q + 1)
    if n_qubits > 2:
        qc.cx(n_qubits - 1, 0)

qc.decompose().draw("text")